# Regional Obligations Profiling
Owner: Joy
Table: `pondolens.dev_bronze.regional_expenditure_obligations`
Baseline: 1,785 rows

## Goal
Identify data issues and propose Silver treatments. Do not modify Bronze.

## Checks
- Check counts by file/year, column types, nulls/blanks, and `_rescued_data`.
- Define what one row represents; check exact duplicates and repeated business keys.
- Verify regions, years, categories, particulars, numeric amounts, units, and price basis.
- Confirm reporting periods and actual/estimate labels.
- Identify totals/subtotals to avoid double counting.
- Compare sample amounts and totals with the original source.
- With Anji, check whether obligations and GAA have matching scope, periods, and regions.
- Check regional compatibility with population and GRDP.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| All columns are STRING (except `_ingested_at` = TIMESTAMP) | All 1,785 rows | Cast `expenditure_amount` to BIGINT in Silver; keep others as STRING for categorical use |
| Dashes in `expenditure_amount` | 509 rows have `"-"` — non-numeric | Replace with NULL in Silver (no obligation recorded in source) |
| NULL / blank `expenditure_amount` | 56 rows have NULL or empty string | Keep as NULL in Silver |
| Totals / subtotals rows | 174 rows where `regions` = "Total" or "Nationwide" | Add `geographic_row_type` column: `geographic_total` for Total, `nonregional` for Nationwide/Central Office, `regional` for named regions |
| `particulars` = "All" are aggregate rows | 144 rows with particulars = "All" | Add `financial_row_type` column: `category_total` for particulars = "All", `grand_total` for category = "GRAND TOTAL", `detail` for specific departments/funds |
| `GRAND TOTAL` category | 41 rows | Flag as `grand_total` in `financial_row_type` |
| Uniform `unit_amount` | All 1,785 rows = "thousand pesos" | Remove `unit_amount` column in Silver; standardize to thousand pesos |
| Uniform `price_basis` | All 1,785 rows = "Actual obligations" | Retain as metadata column (confirms measure type) |
| No `_rescued_data` issues | 0 rows with rescued data | No action needed — clean schema parse |
| No exact duplicates | 0 exact duplicate rows | No action needed |
| No business key duplicates | 0 duplicates on (year, category, particulars, regions) | No deduplication needed; key is unique |
| Only 2 years | 2024 (840 rows), 2025 (945 rows) | Limited time coverage; note for trend analysis |
| `Negros Island Region` only in 2025 | 45 rows (vs 87 per other region) | Appears in 2025 only; verify against source and GAA |
| `Nationwide` vs `Total` as separate region values | Both appear as distinct region labels | Verify meaning: "Total" = sum of all regions, "Nationwide" = non-regional allocation? |

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| One row = one obligation amount for a department/fund x region x year | 0 business key duplicates on (year, category, particulars, regions) | Confirmed |
| `"-"` means no obligation recorded for that department-region | 509 rows with `"-"` in both 2024 and 2025 | Pending verification with source |
| `"Total"` region row = sum of all regional rows for that department | 174 Total/Nationwide rows; Total appears for each department | Pending verification with source |
| `"Nationwide"` = non-regional / centrally executed allocation | Distinct from "Total" in the data | Pending verification with source |
| `"Actual obligations"` = realized obligations, not estimates | `price_basis` = "Actual obligations" for all rows | Pending verification with source |
| Obligations and GAA share same scope, periods, and regions | Not yet cross-compared | Pending (with Anji) |
| Regions match population and GRDP table naming | Not yet cross-compared | Pending |

## Semantic Notes
- **Obligation measure:** "Actual obligations" in thousand pesos (all rows).
- **Unit:** Uniform — all amounts in thousand pesos. No conversion needed; cast to BIGINT in Silver.
- **Rows safe to sum:** Only `detail` rows (specific department/fund x named region) can be summed. Do NOT sum across `geographic_total` (Total), `nonregional` (Nationwide, Central Office), or `grand_total` (GRAND TOTAL category) rows to avoid double counting.
- **Obligations are not automatically cash payments.** They represent committed/incurred obligations, not disbursed amounts.
- **A GAA-obligations difference is not automatically unobligated allotments.** The difference may reflect timing, scope, or classification differences — requires source confirmation.
- **`Negros Island Region`** appears only in 2025 (45 rows). Other regions have 87 rows each (both years). This may reflect a 2025 administrative reorganization.

## Silver Treatment Plan
1. **Cast `expenditure_amount`** to BIGINT; replace `"-"` and blank strings with NULL
2. **Add `financial_row_type`** column:
   - `grand_total` — when `category` = "GRAND TOTAL"
   - `category_total` — when `particulars` = "All"
   - `detail` — specific department or fund name
3. **Add `geographic_row_type`** column:
   - `geographic_total` — when `regions` = "Total"
   - `nonregional` — when `regions` in ("Nationwide", "Central Office")
   - `regional` — named regions (Region I-XII, NCR, CAR, CARAGA, BARMM, Negros Island Region)
4. **Remove `unit_amount`** column (uniform "thousand pesos")
5. **Retain `price_basis`** as metadata (all rows = "Actual obligations")
6. **Keep all rows** in Silver — preserve totals for validation while allowing dashboards to select detail rows without double counting
7. **Standardize `year`** to INTEGER for filtering and joins

## Conclusion
- **Required Silver changes:** Cast expenditure_amount to BIGINT (replace dashes/blanks with NULL); add `financial_row_type` and `geographic_row_type` classification columns; remove `unit_amount`; cast `year` to INTEGER.
- **Ready for regional comparison? Why / why not:** Partially. The data is clean (no duplicates, no rescued data, consistent units, unique business keys). However, cross-table validation with GAA (scope, periods, regions) and population/GRDP compatibility is still pending. The "Nationwide" vs "Total" distinction and "Negros Island Region" (2025 only) need reconciliation with comparison tables.
- **Open questions:**
  - Do obligations and GAA have matching categories, particulars, and regions? (with Anji)
  - Are population and GRDP tables using the same region naming and coverage?
  - What does "Nationwide" mean vs "Total" — is it non-regional allocation?
  - Why does Negros Island Region appear only in 2025?
  - Should `"-"` values be treated as zero or truly NULL?

In [0]:
%sql
-- 1. Preview sample data
SELECT *
FROM pondolens.dev_bronze.regional_expenditure_obligations
LIMIT 10;

In [0]:
%sql
-- 2. Column types and schema
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_obligations;

In [0]:
%sql
-- 3. Row counts by source file and year
SELECT
    _source_file,
    year,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.regional_expenditure_obligations
GROUP BY _source_file, year
ORDER BY _source_file, year;

In [0]:
%sql
-- 4. Null and blank counts per column
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(source_name IS NULL OR source_name = '') AS source_name_nulls,
    COUNT_IF(source_page IS NULL OR source_page = '') AS source_page_nulls,
    COUNT_IF(price_basis IS NULL OR price_basis = '') AS price_basis_nulls,
    COUNT_IF(year IS NULL OR year = '') AS year_nulls,
    COUNT_IF(category IS NULL OR category = '') AS category_nulls,
    COUNT_IF(particulars IS NULL OR particulars = '') AS particulars_nulls,
    COUNT_IF(regions IS NULL OR regions = '') AS regions_nulls,
    COUNT_IF(expenditure_amount IS NULL OR expenditure_amount = '') AS expenditure_amount_nulls,
    COUNT_IF(unit_amount IS NULL OR unit_amount = '') AS unit_amount_nulls,
    COUNT_IF(_rescued_data IS NULL OR _rescued_data = '') AS _rescued_data_nulls,
    COUNT_IF(_source_file IS NULL OR _source_file = '') AS source_file_nulls,
    COUNT_IF(_ingested_at IS NULL) AS ingested_at_nulls
FROM pondolens.dev_bronze.regional_expenditure_obligations;

In [0]:
%sql
-- 5. _rescued_data check (schema parse failures from Auto Loader)
SELECT
    COUNT(*) AS rows_with_rescued_data,
    COUNT(DISTINCT _rescued_data) AS distinct_rescued_values
FROM pondolens.dev_bronze.regional_expenditure_obligations
WHERE _rescued_data IS NOT NULL AND _rescued_data <> '';

In [0]:
%sql
-- 6. Distinct values for key categorical columns
SELECT 'regions' AS column_name, regions AS value, COUNT(*) AS cnt
FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY regions
UNION ALL
SELECT 'year', year, COUNT(*) FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY year
UNION ALL
SELECT 'category', category, COUNT(*) FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY category
UNION ALL
SELECT 'price_basis', price_basis, COUNT(*) FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY price_basis
UNION ALL
SELECT 'unit_amount', unit_amount, COUNT(*) FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY unit_amount
UNION ALL
SELECT 'particulars', particulars, COUNT(*) FROM pondolens.dev_bronze.regional_expenditure_obligations GROUP BY particulars
ORDER BY column_name, cnt DESC;

In [0]:
%sql
-- 7. Expenditure amount validation: non-numeric values, dashes, negatives
SELECT
    expenditure_amount,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.regional_expenditure_obligations
WHERE expenditure_amount IS NOT NULL
  AND expenditure_amount <> ''
  AND expenditure_amount NOT RLIKE '^[0-9]+$'
GROUP BY expenditure_amount
ORDER BY row_count DESC;

In [0]:
%sql
-- 8. Exact duplicates (all business columns)
SELECT
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount,
    COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.regional_expenditure_obligations
GROUP BY
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount
HAVING COUNT(*) > 1;

In [0]:
%sql
-- 9. Business key duplicates: year + category + particulars + regions
-- Repeats may indicate subtotals, multiple source pages, or data errors
SELECT
    year,
    category,
    particulars,
    regions,
    COUNT(*) AS occurrence_count,
    COUNT(DISTINCT expenditure_amount) AS distinct_amounts
FROM pondolens.dev_bronze.regional_expenditure_obligations
GROUP BY year, category, particulars, regions
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

In [0]:
%sql
-- 10. Identify potential totals/subtotals to avoid double counting
-- Rows where regions is 'Total', 'Nationwide', or similar aggregate labels
SELECT
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount,
    source_page
FROM pondolens.dev_bronze.regional_expenditure_obligations
WHERE regions RLIKE '(?i)total'
   OR regions = 'Nationwide'
ORDER BY year, category, source_page;

# Silver Treatment Plan

Based on profiling results for `pondolens.dev_bronze.regional_expenditure_obligations` (1,785 rows).

## Transformations

| # | Column / Action | Treatment | Rationale |
|---|---|---|---|
| 1 | `expenditure_amount` | Cast to `BIGINT`; replace `-` and blank with `NULL` | 509 rows have `-`, 56 rows have NULL/blank — non-numeric strings blocking aggregation |
| 2 | `year` | Cast to `INT` | Currently STRING; needed for filtering and joins |
| 3 | New: `financial_row_type` | `grand_total` when `category = 'GRAND TOTAL'`; `category_total` when `particulars = 'All'`; `detail` for specific departments/funds | 144 aggregate rows + 41 grand total rows must be flagged to avoid double counting |
| 4 | New: `geographic_row_type` | `geographic_total` when `regions = 'Total'`; `nonregional` when `regions IN ('Nationwide', 'Central Office')`; `regional` for named regions | 174 Total/Nationwide rows are aggregate or non-regional — must be excluded when summing regional detail |
| 5 | `unit_amount` | **Drop** | All 1,785 rows = "thousand pesos" — no variance, redundant |
| 6 | `price_basis` | **Keep** as metadata | All rows = "Actual obligations" — confirms measure type; retain for documentation |
| 7 | `source_name`, `source_page`, `_source_file`, `_ingested_at` | **Keep** as provenance columns | Needed for traceability back to source PDF/CSV |
| 8 | `category`, `particulars`, `regions` | **Keep** as-is (STRING) | Categorical dimensions for grouping and filtering |
| 9 | All rows | **Keep all rows** in Silver | Preserve totals for validation; dashboards select `detail` + `regional` rows only to avoid double counting |

## Silver Column Summary

| Column | Type | Source |
|---|---|---|
| `year` | INT | Cast from bronze `year` |
| `category` | STRING | Direct from bronze |
| `particulars` | STRING | Direct from bronze |
| `regions` | STRING | Direct from bronze |
| `expenditure_amount` | BIGINT | Cast from bronze (NULL for `-`/blank) |
| `price_basis` | STRING | Direct from bronze (metadata) |
| `financial_row_type` | STRING | Derived from `category` and `particulars` |
| `geographic_row_type` | STRING | Derived from `regions` |
| `source_name` | STRING | Direct from bronze (provenance) |
| `source_page` | STRING | Direct from bronze (provenance) |
| `_source_file` | STRING | Direct from bronze (provenance) |
| `_ingested_at` | TIMESTAMP | Direct from bronze (provenance) |

## Rules for Safe Aggregation

- **Sum only** rows where `financial_row_type = 'detail'` AND `geographic_row_type = 'regional'`
- **Never sum** across `grand_total`, `category_total`, or `geographic_total` rows
- `nonregional` rows (Nationwide, Central Office) are valid detail but not regional — sum separately if needed
- A `GRAND TOTAL` row should equal the sum of all `DEPARTMENTS` + `SPECIAL PURPOSE FUNDS` + `AUTOMATIC APPROPRIATIONS` `Total` rows — use for validation only